# Vector Search index
A Delta Sync index on `docs_chunks` with embeddings computed by Databricks (`databricks-gte-large-en`). The endpoint is a bundle resource. The index is created here because it needs the source table to exist, which is only true after the first run. Later runs trigger a sync, which embeds only the rows changed since the last sync.

Access: readers get `SELECT` on the chunks and the index, nothing on the raw volume or on the logs.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

dbutils.widgets.text("vs_endpoint", "")
dbutils.widgets.text("embedding_endpoint", "")
dbutils.widgets.text("uc_readers", "")

import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    EndpointStatusState,
    PipelineType,
    VectorIndexType,
)

w = WorkspaceClient()
vs_endpoint = dbutils.widgets.get("vs_endpoint")
source_table = f"{target}.docs_chunks"
index_name = f"{target}.docs_chunks_index"

In [ ]:
deadline = time.time() + 1800
while w.vector_search_endpoints.get_endpoint(vs_endpoint).endpoint_status.state != EndpointStatusState.ONLINE:
    if time.time() > deadline:
        raise TimeoutError(f"endpoint {vs_endpoint} is not online")
    time.sleep(30)

try:
    w.vector_search_indexes.get_index(index_name)
    w.vector_search_indexes.sync_index(index_name)
    print("sync started")
except NotFound:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=vs_endpoint,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(name="content", embedding_model_endpoint_name=dbutils.widgets.get("embedding_endpoint"))
            ],
            columns_to_sync=["chunk_id", "chunk_profile", "doc_path", "doc_title", "section", "content", "source_url"],
        ),
    )
    print("index created")

In [ ]:
expected = spark.table(source_table).count()
version = spark.sql(f"DESCRIBE HISTORY {source_table} LIMIT 1").first().version
deadline = time.time() + 3600
while True:
    status = w.vector_search_indexes.get_index(index_name).status
    if status.ready and status.indexed_row_count == expected and "progress" not in (status.message or "").lower():
        break
    if time.time() > deadline:
        raise TimeoutError(f"index not synced: {status.message}")
    time.sleep(30)
print(f"{status.indexed_row_count} rows indexed, source version {version}")

In [ ]:
readers = dbutils.widgets.get("uc_readers")
spark.sql(f"GRANT SELECT ON TABLE {source_table} TO `{readers}`")
spark.sql(f"GRANT SELECT ON TABLE {index_name} TO `{readers}`")
display(spark.sql(f"SHOW GRANTS ON TABLE {index_name}"))